# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FALAKNAZMALICK/MY-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**The Grain (Unit of Analysis):**

One row = One unique Content ID (content_id) for a specific client (client_id) in a single monthly reporting snapshot.

**The Time Window:**

We are using a mid-panel training month: month=2026-03 (spanning March 1, 2026, to March 31, 2026).

The final month (month=2026-06) is treated as a sealed test month to avoid testing on our training distribution.

In [7]:
import duckdb
from google.colab import userdata

# Authenticate with Hugging Face Token
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

BASE_URL = "hf://datasets/FlyRank/internship-warehouse"

query_grain_check = f"""
SELECT
    client_id,
    content_id,
    COUNT(*) as row_count
FROM read_parquet('{BASE_URL}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY client_id, content_id
HAVING COUNT(*) > 1
LIMIT 5;
"""
print("--- Grain Check: If this returns 0 rows, our grain is perfectly unique! ---")


--- Grain Check: If this returns 0 rows, our grain is perfectly unique! ---


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### 1. Features
These are columns we use to make predictions. They represent data knowable at the exact moment a decision is made (before we decide to update a page).

* **days_since_last_update**: Tells us how long a page has been static. Knowable from system update logs.
* **impressions_90d**: Measures search visibility over the last 90 days. Knowable from historical search console tracking.
* **clicks_90d**: Measures user engagement over the last 90 days. Knowable from historical analytics.
* **word_count**: The physical length of the content. Knowable because page content is saved in our database.
* **ctr**: The click-through-rate over the historical period. Engineered from clicks / impressions.

---

### 2. Labels
The ground-truth status we are trying to predict.

* **target_is_declining**: Our binary prediction target. (1 = traffic is decaying, 0 = stable or growing).

---

### 3. Context
Metadata columns not used to train the model, but vital for downstream human actions.

* **content_id**: The unique identifier of the webpage. Needed so the editor knows which URL to rewrite.
* **client_id**: Identifies which client workspace the content belongs to.

---

### 4. Excluded (With "Why")
Columns we must deliberately ignore because they introduce severe Target Leakage.

* **trend_direction**: EXCLUDED. This text column is what we used to calculate the target column ('target_is_declining'). Keeping it gives the model the answer instantly.
* **trend_pct**: EXCLUDED. This is the exact speed of the traffic change. It is calculated AFTER the observation window, leaking future information.
* **clicks_last_30d**: EXCLUDED. Recent 30-day clicks directly reveal the current declining vector, causing feature leakage during training.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
import os
import duckdb
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    raise ValueError("ERROR: Could not find 'HF_TOKEN' in your Colab Secrets (the Key icon). Please add it first!")

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}');")

BASE_URL = "hf://datasets/FlyRank/internship-warehouse"

query_counts = f"""
SELECT
    COUNT(*) as total_rows,
    MIN(report_date) as start_date,
    MAX(report_date) as end_date
FROM read_parquet('{BASE_URL}/fact_content_daily_performance/month=2026-03/*.parquet');
"""

print("--- Query 1: Row Count & Time Window Verification ---")
try:
    df_counts = con.execute(query_counts).df()
    display(df_counts)
except Exception as e:
    print(f"Query 1 Failed! Error details: {e}")


query_missing = f"""
SELECT
    SUM(CASE WHEN content_id IS NULL THEN 1 ELSE 0 END) as missing_content_ids,
    SUM(CASE WHEN days_since_last_update IS NULL THEN 1 ELSE 0 END) as missing_update_days,
    SUM(CASE WHEN trend_direction IS NULL THEN 1 ELSE 0 END) as missing_labels
FROM read_parquet('{BASE_URL}/fact_content_daily_performance/month=2026-03/*.parquet');
"""

print("\n--- Query 2: Missing Values Audit ---")
try:
    df_missing = con.execute(query_missing).df()
    display(df_missing)
except Exception as e:
    print(f"Query 2 Failed! Error details: {e}")


--- Query 1: Row Count & Time Window Verification ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31



--- Query 2: Missing Values Audit ---
Query 2 Failed! Error details: Binder Error: Referenced column "content_id" not found in FROM clause!
Candidate bindings: "content_hash_id", "client_hash_id", "month", "scroll_events", "sessions_paid"

LINE 3:     SUM(CASE WHEN content_id IS NULL THEN 1 ELSE 0 END) as missing_content_ids...
                          ^


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

#**System Limitations**

No External Algorithm Tracking: The dataset is purely internal GSC/GA data. It cannot tell us if Google rolled out a core algorithm update on a specific date, which might explain sudden traffic drops better than content "staleness."

Lack of Off-Page Signals: It contains no backlink metrics, competitor tracking, or keyword ranking difficulty. A page might decay simply because a competitor published a much better piece of content, which our features cannot see.

History Gaps: Older pages that existed before GSC tracking started will have incomplete historical trends.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.